In [1]:
import numpy as np

# Task times and facility capacities
durations = [5, 8, 4, 7, 6, 3, 9]
limits = [24, 30, 28]

# Cost matrix for assigning each task to each facility
assignment_cost = [
    [10, 12, 9],
    [15, 14, 16],
    [8, 9, 7],
    [12, 10, 13],
    [14, 13, 12],
    [9, 8, 10],
    [11, 12, 13]
]

# GA Configurations
pop_size = 6
generations = 100
cross_prob = 0.8
mutate_prob = 0.2

def evaluate(solution):
    time_used = [0] * 3
    total_cost = 0
    extra_penalty = 0

    for t_idx, facility in enumerate(solution):
        f = facility - 1
        time_used[f] += durations[t_idx]
        total_cost += durations[t_idx] * assignment_cost[t_idx][f]

    for i in range(3):
        if time_used[i] > limits[i]:
            extra_penalty += (time_used[i] - limits[i]) * 1000

    return 1 / (total_cost + extra_penalty)

def roulette_selection(pop, fitness_vals):
    total_fit = sum(fitness_vals)
    chances = [f / total_fit for f in fitness_vals]
    pick = np.random.choice(len(pop), p=chances)
    return pop[pick]

def crossover(parent_a, parent_b):
    if np.random.rand() < cross_prob:
        split = np.random.randint(1, len(parent_a))
        return parent_a[:split] + parent_b[split:], parent_b[:split] + parent_a[split:]
    return parent_a[:], parent_b[:]

def mutate(ind):
    if np.random.rand() < mutate_prob:
        idx1, idx2 = np.random.choice(len(ind), 2, replace=False)
        ind[idx1], ind[idx2] = ind[idx2], ind[idx1]
    return ind

# Initial random population
group = []
for _ in range(pop_size):
    ind = list(np.random.randint(1, 4, size=len(durations)))
    group.append(ind)

# Evolution loop
for gen in range(generations):
    fit_vals = [evaluate(ind) for ind in group]
    next_group = []

    for _ in range(pop_size // 2):
        p1 = roulette_selection(group, fit_vals)
        p2 = roulette_selection(group, fit_vals)
        off1, off2 = crossover(p1, p2)
        next_group.append(mutate(off1))
        next_group.append(mutate(off2))

    group = next_group

# Find best in final generation
final_scores = [evaluate(ind) for ind in group]
best_index = np.argmax(final_scores)
best_sol = group[best_index]

print("Best task assignment to facilities:", best_sol)

# Calculate final cost
final_cost = sum(durations[i] * assignment_cost[i][best_sol[i] - 1] for i in range(len(durations)))
print("Final total cost:", final_cost)


Best task assignment to facilities: [2, 3, 2, 2, 3, 2, 2]
Final total cost: 498
